# 03 — Modelagem com Random Forest Regressor

Este notebook treina modelos de **Random Forest Regressor** usando **PySpark MLlib** para prever temperatura no estado de São Paulo.

Serão treinados dois modelos:

1. **Previsão da temperatura média de amanhã**.
2. **Previsão da temperatura média dos próximos 7 dias**.

As bases utilizadas aqui já foram geradas no notebook de pré-processamento e estão salvas em formato **Parquet**. Elas já passaram por:

- limpeza e padronização dos dados;
- tratamento de valores sentinela;
- aplicação de regras físicas;
- criação de variáveis temporais e geográficas;
- agregação diária por estação;
- criação dos alvos de previsão;
- separação temporal entre treino e teste;
- imputação sem vazamento de dados;
- indexação das variáveis categóricas.

A separação treino/teste segue uma lógica temporal:

- **Treino:** anos anteriores a 2018;
- **Teste:** anos de 2018 em diante.

Além das métricas gerais, este notebook também gera gráficos com **Plotly** e analisa o comportamento do modelo por tipo de área, com foco em:

- urbano/metropolitano;
- litoral;
- serra/altitude;
- interior.

## 1. Imports e inicialização da SparkSession

Nesta etapa são importadas as bibliotecas necessárias para:

- carregar os Parquets;
- montar o vetor de features;
- treinar o modelo Random Forest com MLlib;
- avaliar regressão;
- gerar gráficos interativos com Plotly.

In [ ]:
from pyspark.sql import SparkSession

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
spark = (
    SparkSession.builder
    .appName("03_modelagem_random_forest")
    .getOrCreate()
)

spark

## 2. Caminhos das bases Parquet

As bases finais já foram criadas no notebook de pré-processamento.

Aqui serão carregados os datasets separados de treino e teste para os dois objetivos:

- previsão da temperatura média de amanhã;
- previsão da temperatura média dos próximos 7 dias.

Também serão carregadas as bases completas, porque elas ainda possuem colunas interpretáveis como `tipo_area`, `macro_regiao_sp` e `faixa_altitude`. Essas colunas serão usadas depois para análises e gráficos por tipo de região.

In [ ]:
amanha_train_path = "/home/jovyan/work/data/processed/weather_sp_amanha_train"
amanha_test_path  = "/home/jovyan/work/data/processed/weather_sp_amanha_test"

semana_train_path = "/home/jovyan/work/data/processed/weather_sp_semana_train"
semana_test_path  = "/home/jovyan/work/data/processed/weather_sp_semana_test"

dataset_amanha_path = "/home/jovyan/work/data/processed/weather_sp_dataset_amanha"
dataset_semana_path = "/home/jovyan/work/data/processed/weather_sp_dataset_semana"

modelos_path = "/home/jovyan/work/models"
resultados_path = "/home/jovyan/work/data/processed"

## 3. Carregamento das bases

As bases são lidas diretamente do formato Parquet.

In [ ]:
amanha_train = spark.read.parquet(amanha_train_path)
amanha_test = spark.read.parquet(amanha_test_path)

semana_train = spark.read.parquet(semana_train_path)
semana_test = spark.read.parquet(semana_test_path)

dataset_amanha_completo = spark.read.parquet(dataset_amanha_path)
dataset_semana_completo = spark.read.parquet(dataset_semana_path)

In [ ]:
print("Dataset amanhã")
print(f"Treino: {amanha_train.count():,}")
print(f"Teste : {amanha_test.count():,}")

print("\nDataset próximos 7 dias")
print(f"Treino: {semana_train.count():,}")
print(f"Teste : {semana_test.count():,}")

## 4. Cache estratégico dos datasets de modelagem

No pré-processamento, o cache foi evitado para reduzir consumo de memória.

Aqui, no notebook de modelagem, o cache faz sentido porque os mesmos datasets serão usados várias vezes:

- treinamento;
- avaliação;
- geração de previsões;
- análise de erro;
- gráficos.

O `count()` logo após o `cache()` força a materialização dos dados em memória.

In [ ]:
amanha_train = amanha_train.cache()
amanha_test = amanha_test.cache()

semana_train = semana_train.cache()
semana_test = semana_test.cache()

amanha_train.count()
amanha_test.count()
semana_train.count()
semana_test.count()

## 5. Conferência dos schemas

Antes de montar o modelo, é importante verificar se os datasets possuem:

- colunas de identificação;
- features numéricas já imputadas;
- variáveis categóricas indexadas;
- coluna-alvo correta.

In [ ]:
amanha_train.printSchema()

In [ ]:
semana_train.printSchema()

## 6. Definição dos alvos e das features

O modelo Random Forest do MLlib recebe as variáveis explicativas em uma única coluna vetorial chamada `features`.

Como o pré-processamento já deixou as colunas prontas para modelagem, as features serão identificadas automaticamente, removendo apenas:

- colunas de identificação;
- coluna-alvo.

As colunas de identificação não entram no modelo porque representam nomes, códigos ou datas, e não variáveis numéricas explicativas diretas.

In [ ]:
coluna_alvo_amanha = "temperatura_amanha"
coluna_alvo_semana = "temperatura_media_proximos_7_dias"

amanha_train.createOrReplaceTempView("amanha_train")
semana_train.createOrReplaceTempView("semana_train")

colunas_amanha = spark.sql("DESCRIBE amanha_train")
colunas_semana = spark.sql("DESCRIBE semana_train")

colunas_amanha.createOrReplaceTempView("colunas_amanha")
colunas_semana.createOrReplaceTempView("colunas_semana")

features_amanha_df = spark.sql(f"""
    SELECT col_name
    FROM colunas_amanha
    WHERE col_name NOT IN (
        'station',
        'station_code',
        'data_formatada',
        '{coluna_alvo_amanha}'
    )
    AND col_name <> ''
    AND col_name NOT LIKE '#%'
""")

features_semana_df = spark.sql(f"""
    SELECT col_name
    FROM colunas_semana
    WHERE col_name NOT IN (
        'station',
        'station_code',
        'data_formatada',
        '{coluna_alvo_semana}'
    )
    AND col_name <> ''
    AND col_name NOT LIKE '#%'
""")

features_amanha = [
    row["col_name"]
    for row in features_amanha_df.collect()
]

features_semana = [
    row["col_name"]
    for row in features_semana_df.collect()
]

print(f"Features amanhã ({len(features_amanha)}):")
print(features_amanha)

print(f"\nFeatures semana ({len(features_semana)}):")
print(features_semana)

## 7. Por que Random Forest?

O **Random Forest** é um modelo baseado em múltiplas árvores de decisão.

Ele é interessante para este projeto porque:

- captura relações não lineares;
- funciona bem com variáveis meteorológicas;
- lida bem com interações entre temperatura, umidade, pressão, altitude e sazonalidade;
- permite analisar a importância das variáveis;
- é mais robusto do que uma única árvore de decisão.

Um ponto importante: o Random Forest **não entende sequência temporal sozinho**. Por isso, o pré-processamento criou features de defasagem e janelas móveis, como:

- temperatura média de ontem;
- média dos últimos 3 dias;
- média dos últimos 7 dias;
- umidade média dos últimos 7 dias;
- precipitação acumulada dos últimos 7 dias.

Assim, o modelo recebe contexto temporal em forma de colunas explicativas.

## 8. Treinamento do modelo para temperatura de amanhã

Nesta etapa será treinado o primeiro modelo:

> prever a `temperatura_amanha`.

O `VectorAssembler` junta todas as features em uma coluna vetorial, e o `RandomForestRegressor` realiza o treinamento.

In [ ]:
assembler_amanha = VectorAssembler(
    inputCols=features_amanha,
    outputCol="features",
    handleInvalid="keep"
)

rf_amanha = RandomForestRegressor(
    featuresCol="features",
    labelCol=coluna_alvo_amanha,
    predictionCol="prediction",
    numTrees=50,
    maxDepth=10,
    minInstancesPerNode=5,
    seed=42
)

pipeline_amanha = Pipeline(stages=[
    assembler_amanha,
    rf_amanha
])

In [ ]:
modelo_rf_amanha = pipeline_amanha.fit(amanha_train)

pred_amanha = modelo_rf_amanha.transform(amanha_test)

print("Modelo Random Forest para temperatura de amanhã treinado com sucesso.")

## 9. Previsões do modelo de amanhã

Após o treinamento, o modelo é aplicado à base de teste.

Também é criada a coluna `erro_absoluto`, que representa a diferença absoluta entre o valor real e o valor previsto.

In [ ]:
pred_amanha.createOrReplaceTempView("pred_amanha_raw")

pred_amanha = spark.sql(f"""
    SELECT
        *,
        ABS({coluna_alvo_amanha} - prediction) AS erro_absoluto
    FROM pred_amanha_raw
""")

pred_amanha.createOrReplaceTempView("pred_amanha")

spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND({coluna_alvo_amanha}, 2) AS real,
        ROUND(prediction, 2) AS previsto,
        ROUND(erro_absoluto, 2) AS erro_abs
    FROM pred_amanha
""").show(20, truncate=False)

## 10. Função de avaliação de regressão

Serão usadas três métricas:

- **MAE:** erro médio absoluto em °C. É a métrica mais fácil de interpretar.
- **RMSE:** penaliza erros grandes com mais intensidade.
- **R²:** indica a proporção da variação explicada pelo modelo.

Quanto menores MAE e RMSE, melhor. Quanto mais próximo de 1 o R², melhor.

In [ ]:
def avaliar_regressao(predicoes, coluna_alvo, nome_modelo):
    avaliador_mae = RegressionEvaluator(
        labelCol=coluna_alvo,
        predictionCol="prediction",
        metricName="mae"
    )

    avaliador_rmse = RegressionEvaluator(
        labelCol=coluna_alvo,
        predictionCol="prediction",
        metricName="rmse"
    )

    avaliador_r2 = RegressionEvaluator(
        labelCol=coluna_alvo,
        predictionCol="prediction",
        metricName="r2"
    )

    mae = avaliador_mae.evaluate(predicoes)
    rmse = avaliador_rmse.evaluate(predicoes)
    r2 = avaliador_r2.evaluate(predicoes)

    print(nome_modelo)
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

    return {
        "modelo": nome_modelo,
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2)
    }

In [ ]:
metricas_amanha = avaliar_regressao(
    pred_amanha,
    coluna_alvo_amanha,
    "Random Forest - Temperatura amanhã"
)

## 11. Treinamento do modelo para média dos próximos 7 dias

Agora será treinado o segundo modelo:

> prever a `temperatura_media_proximos_7_dias`.

Este alvo representa a temperatura média dos sete dias seguintes para cada estação meteorológica.

In [ ]:
assembler_semana = VectorAssembler(
    inputCols=features_semana,
    outputCol="features",
    handleInvalid="keep"
)

rf_semana = RandomForestRegressor(
    featuresCol="features",
    labelCol=coluna_alvo_semana,
    predictionCol="prediction",
    numTrees=50,
    maxDepth=10,
    minInstancesPerNode=5,
    seed=42
)

pipeline_semana = Pipeline(stages=[
    assembler_semana,
    rf_semana
])

In [ ]:
modelo_rf_semana = pipeline_semana.fit(semana_train)

pred_semana = modelo_rf_semana.transform(semana_test)

print("Modelo Random Forest para temperatura média dos próximos 7 dias treinado com sucesso.")

## 12. Previsões do modelo de próximos 7 dias

In [ ]:
pred_semana.createOrReplaceTempView("pred_semana_raw")

pred_semana = spark.sql(f"""
    SELECT
        *,
        ABS({coluna_alvo_semana} - prediction) AS erro_absoluto
    FROM pred_semana_raw
""")

pred_semana.createOrReplaceTempView("pred_semana")

spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND({coluna_alvo_semana}, 2) AS real,
        ROUND(prediction, 2) AS previsto,
        ROUND(erro_absoluto, 2) AS erro_abs
    FROM pred_semana
""").show(20, truncate=False)

In [ ]:
metricas_semana = avaliar_regressao(
    pred_semana,
    coluna_alvo_semana,
    "Random Forest - Temperatura média próximos 7 dias"
)

## 13. Comparação final das métricas

Nesta etapa, as métricas dos dois modelos são reunidas em uma única tabela.

Essa comparação ajuda a entender se o modelo tem melhor desempenho prevendo o dia seguinte ou a média dos próximos sete dias.

In [ ]:
metricas_rf = spark.createDataFrame([
    metricas_amanha,
    metricas_semana
])

metricas_rf.show(truncate=False)

### Gráfico: comparação do MAE

O MAE é especialmente útil para comunicação do resultado, pois pode ser lido diretamente como erro médio em graus Celsius.

In [ ]:
metricas_rf.createOrReplaceTempView("metricas_rf")

metricas_plot = spark.sql("""
    SELECT
        modelo,
        mae
    FROM metricas_rf
    ORDER BY modelo
""").collect()

modelos = [linha["modelo"] for linha in metricas_plot]
maes = [linha["mae"] for linha in metricas_plot]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=modelos,
        y=maes,
        text=maes,
        texttemplate="%{text:.2f}",
        textposition="outside"
    )
)

fig.update_layout(
    title="Comparação do MAE entre os modelos Random Forest",
    xaxis_title="Modelo",
    yaxis_title="MAE (°C)"
)

fig.show()

### Insight esperado

Se o modelo de próximos 7 dias apresentar MAE menor do que o modelo de amanhã, isso pode indicar que médias semanais suavizam oscilações diárias.

Em outras palavras, prever uma média de 7 dias pode ser mais estável do que prever exatamente o comportamento do dia seguinte, que pode sofrer influência de mudanças bruscas de tempo.

## 14. Importância das variáveis

O Random Forest permite observar a importância relativa de cada feature usada no treinamento.

Essa análise ajuda a responder perguntas como:

- quais variáveis mais influenciam a previsão?
- as variáveis de temperatura recente são mais importantes?
- altitude, latitude e longitude ajudam a diferenciar regiões climáticas?
- as janelas temporais criadas no pré-processamento foram úteis?

In [ ]:
def obter_importancias(modelo_pipeline, lista_features, nome_view):
    modelo_rf = modelo_pipeline.stages[-1]
    importancias = modelo_rf.featureImportances

    valores_sql = []

    for feature, importancia in zip(lista_features, importancias):
        feature_sql = feature.replace("'", "''")
        valores_sql.append(f"('{feature_sql}', {float(importancia)})")

    valores_sql = ",\n        ".join(valores_sql)

    importancias_df = spark.sql(f"""
        SELECT *
        FROM VALUES
            {valores_sql}
        AS importancias(feature, importancia)
    """)

    importancias_df.createOrReplaceTempView(nome_view)

    return importancias_df

In [ ]:
importancias_amanha = obter_importancias(
    modelo_rf_amanha,
    features_amanha,
    "importancias_amanha"
)

importancias_semana = obter_importancias(
    modelo_rf_semana,
    features_semana,
    "importancias_semana"
)

### Gráfico: top 15 variáveis mais importantes — amanhã

In [ ]:
importancias_amanha.createOrReplaceTempView("importancias_amanha")

imp_amanha_rows = spark.sql("""
    SELECT
        feature,
        importancia
    FROM importancias_amanha
    ORDER BY importancia ASC
    LIMIT 15
""").collect()

imp_amanha_plot = {
    "feature": [row["feature"] for row in imp_amanha_rows],
    "importancia": [row["importancia"] for row in imp_amanha_rows]
}

fig = px.bar(
    imp_amanha_plot,
    x="importancia",
    y="feature",
    orientation="h",
    title="Top 15 variáveis mais importantes — previsão de amanhã",
    labels={
        "importancia": "Importância",
        "feature": "Variável"
    }
)

fig.show()

### Gráfico: top 15 variáveis mais importantes — próximos 7 dias

In [ ]:
importancias_semana.createOrReplaceTempView("importancias_semana")

imp_semana_rows = spark.sql("""
    SELECT
        feature,
        importancia
    FROM (
        SELECT
            feature,
            importancia
        FROM importancias_semana
        ORDER BY importancia DESC
        LIMIT 15
    )
    ORDER BY importancia ASC
""").collect()

imp_semana_plot = {
    "feature": [row["feature"] for row in imp_semana_rows],
    "importancia": [row["importancia"] for row in imp_semana_rows]
}

fig = px.bar(
    imp_semana_plot,
    x="importancia",
    y="feature",
    orientation="h",
    title="Top 15 variáveis mais importantes — previsão dos próximos 7 dias",
    labels={
        "importancia": "Importância",
        "feature": "Variável"
    }
)

fig.show()

### Insight esperado

As variáveis de temperatura recente tendem a aparecer entre as mais importantes, como:

- `temp_media_dia_imputado`;
- `temp_media_ontem_imputado`;
- `temp_media_ultimos_3_dias_imputado`;
- `temp_media_ultimos_7_dias_imputado`.

Isso é coerente com o comportamento climático: a temperatura futura depende fortemente das condições térmicas recentes.

Também é esperado que variáveis como altitude, latitude, longitude e mês tenham alguma importância, pois representam fatores geográficos e sazonais que influenciam a temperatura no estado de São Paulo.

## 15. Recuperação das informações geográficas interpretáveis

As bases finais de treino e teste foram salvas com as features já prontas para modelagem.

Para criar gráficos e insights mais interpretáveis, vamos recuperar da base completa algumas colunas geográficas:

- `tipo_area`;
- `macro_regiao_sp`;
- `faixa_altitude`;
- `altitude`;
- `latitude`;
- `longitude`.

Essa é a opção escolhida para permitir análises por urbano/metropolitano, litoral e serra/altitude sem mexer novamente no pré-processamento.

In [ ]:
dataset_amanha_completo.createOrReplaceTempView("dataset_amanha_completo")
dataset_semana_completo.createOrReplaceTempView("dataset_semana_completo")

contexto_amanha = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        altitude_imputado,
        latitude_imputado,
        longitude_imputado
    FROM dataset_amanha_completo
""")

contexto_semana = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        altitude_imputado,
        latitude_imputado,
        longitude_imputado
    FROM dataset_semana_completo
""")

In [ ]:
pred_amanha.createOrReplaceTempView("pred_amanha")
pred_semana.createOrReplaceTempView("pred_semana")

pred_amanha_ctx = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        altitude_imputado,
        latitude_imputado,
        longitude_imputado,
        temperatura_amanha,
        prediction,
        erro_absoluto
    FROM pred_amanha
""")

pred_semana_ctx = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        altitude_imputado,
        latitude_imputado,
        longitude_imputado,
        temperatura_media_proximos_7_dias,
        prediction,
        erro_absoluto
    FROM pred_semana
""")

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

spark.sql(f"""
    SELECT
        station,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        ROUND({coluna_alvo_amanha}, 2) AS real,
        ROUND(prediction, 2) AS previsto,
        ROUND(erro_absoluto, 2) AS erro_abs
    FROM pred_amanha_ctx
""").show(20, truncate=False)

## 16. Erro médio por tipo de área

Nesta seção, o erro do modelo é analisado por tipo de área.

Essa análise é importante porque diferentes regiões podem ter comportamentos climáticos distintos:

- áreas urbanas podem sofrer influência de ilha de calor;
- áreas litorâneas tendem a ter maior influência da umidade e do oceano;
- regiões de serra/altitude podem apresentar temperaturas mais baixas e variações específicas;
- áreas do interior podem ter maior amplitude térmica em alguns períodos.

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

erro_tipo_area_amanha = spark.sql(f"""
    SELECT
        tipo_area_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG({coluna_alvo_amanha}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_amanha_ctx
    GROUP BY tipo_area_idx
    ORDER BY tipo_area_idx
""")

erro_tipo_area_amanha.show(truncate=False)

In [ ]:
pred_semana_ctx.createOrReplaceTempView("pred_semana_ctx")

erro_tipo_area_semana = spark.sql(f"""
    SELECT
        tipo_area_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG({coluna_alvo_semana}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_semana_ctx
    GROUP BY tipo_area_idx
    ORDER BY tipo_area_idx
""")

erro_tipo_area_semana.show(truncate=False)

### Gráfico: MAE por tipo de área — amanhã

In [ ]:
erro_tipo_area_amanha.createOrReplaceTempView("erro_tipo_area_amanha")

erro_tipo_area_amanha_rows = spark.sql("""
    SELECT
        tipo_area_idx,
        mae
    FROM erro_tipo_area_amanha
    ORDER BY tipo_area_idx
""").collect()

erro_tipo_area_amanha_plot = {
    "tipo_area_idx": [row["tipo_area_idx"] for row in erro_tipo_area_amanha_rows],
    "mae": [row["mae"] for row in erro_tipo_area_amanha_rows]
}

fig = px.bar(
    erro_tipo_area_amanha_plot,
    x="tipo_area_idx",
    y="mae",
    text="mae",
    title="Erro médio absoluto por tipo de área — previsão de amanhã",
    labels={
        "tipo_area_idx": "Tipo de área",
        "mae": "MAE (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Gráfico: MAE por tipo de área — próximos 7 dias

In [ ]:
erro_tipo_area_semana.createOrReplaceTempView("erro_tipo_area_semana")

erro_tipo_area_semana_rows = spark.sql("""
    SELECT
        tipo_area_idx,
        mae
    FROM erro_tipo_area_semana
    ORDER BY tipo_area_idx
""").collect()

erro_tipo_area_semana_plot = {
    "tipo_area_idx": [row["tipo_area_idx"] for row in erro_tipo_area_semana_rows],
    "mae": [row["mae"] for row in erro_tipo_area_semana_rows]
}

fig = px.bar(
    erro_tipo_area_semana_plot,
    x="tipo_area_idx",
    y="mae",
    text="mae",
    title="Erro médio absoluto por tipo de área — previsão dos próximos 7 dias",
    labels={
        "tipo_area_idx": "Tipo de área",
        "mae": "MAE (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Se o erro for maior em `serra_altitude`, isso pode indicar que a dinâmica térmica dessas áreas é mais específica por causa do relevo e da altitude.

Se o erro for maior em `litoral`, pode haver influência de umidade, brisa marítima e menor amplitude térmica, que talvez não estejam totalmente representadas pelas features disponíveis.

Se o erro for maior em `urbano_metropolitano`, pode ser um indício de que efeitos locais, como concentração urbana e ilha de calor, são relevantes para a previsão.

## 17. Real vs previsto por tipo de área

Este gráfico compara a temperatura real com a temperatura prevista.

Quanto mais próximos os pontos estiverem da linha ideal, melhor é o desempenho do modelo.

- Pontos acima da linha: o modelo superestimou a temperatura.
- Pontos abaixo da linha: o modelo subestimou a temperatura.

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

comparacao_tipo_area_amanha_rows = spark.sql(f"""
    SELECT
        tipo_area_idx,
        {coluna_alvo_amanha} AS real,
        prediction AS previsto
    FROM pred_amanha_ctx
    WHERE tipo_area_idx IS NOT NULL
      AND {coluna_alvo_amanha} IS NOT NULL
      AND prediction IS NOT NULL
""").collect()

comparacao_tipo_area_amanha_plot = {
    "tipo_area_idx": [row["tipo_area_idx"] for row in comparacao_tipo_area_amanha_rows],
    "real": [row["real"] for row in comparacao_tipo_area_amanha_rows],
    "previsto": [row["previsto"] for row in comparacao_tipo_area_amanha_rows]
}

In [ ]:
fig = px.scatter(
    comparacao_tipo_area_amanha_plot,
    x="real",
    y="previsto",
    color="tipo_area_idx",
    opacity=0.5,
    title="Temperatura real vs prevista por tipo de área — amanhã",
    labels={
        "real": "Temperatura real (°C)",
        "previsto": "Temperatura prevista (°C)",
        "tipo_area_idx": "Tipo de área"
    }
)

min_val = min(
    min(comparacao_tipo_area_amanha_plot["real"]),
    min(comparacao_tipo_area_amanha_plot["previsto"])
)

max_val = max(
    max(comparacao_tipo_area_amanha_plot["real"]),
    max(comparacao_tipo_area_amanha_plot["previsto"])
)

fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        name="Linha ideal"
    )
)

fig.show()

## 18. Erro por mês e tipo de área

A análise mensal mostra se o modelo erra mais em determinados períodos do ano.

Isso é útil para detectar sazonalidade ou meses de maior instabilidade climática.

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx_sem_mes")

pred_amanha_ctx = spark.sql("""
    SELECT
        *,
        MONTH(data_formatada) AS mes
    FROM pred_amanha_ctx_sem_mes
""")

pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

erro_mes_tipo_area_amanha = spark.sql("""
    SELECT
        mes,
        tipo_area_idx,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_amanha_ctx
    GROUP BY mes, tipo_area_idx
    ORDER BY mes, tipo_area_idx
""")

erro_mes_tipo_area_amanha.show(100, truncate=False)

In [ ]:
erro_mes_tipo_area_amanha.createOrReplaceTempView("erro_mes_tipo_area_amanha")

erro_mes_tipo_area_amanha_rows = spark.sql("""
    SELECT
        mes,
        tipo_area_idx,
        mae
    FROM erro_mes_tipo_area_amanha
    ORDER BY mes, tipo_area_idx
""").collect()

erro_mes_tipo_area_amanha_plot = {
    "mes": [row["mes"] for row in erro_mes_tipo_area_amanha_rows],
    "tipo_area_idx": [row["tipo_area_idx"] for row in erro_mes_tipo_area_amanha_rows],
    "mae": [row["mae"] for row in erro_mes_tipo_area_amanha_rows]
}

fig = px.line(
    erro_mes_tipo_area_amanha_plot,
    x="mes",
    y="mae",
    color="tipo_area_idx",
    markers=True,
    title="MAE por mês e tipo de área — previsão de amanhã",
    labels={
        "mes": "Mês",
        "mae": "MAE (°C)",
        "tipo_area_idx": "Tipo de área"
    }
)

fig.show()

### Insight esperado

Se o erro aumentar em meses de transição, como outono e primavera, isso pode indicar que o modelo tem mais dificuldade em períodos de maior variabilidade atmosférica.

Se algum tipo de área apresentar erro sistematicamente maior ao longo do ano, isso sugere que o comportamento climático daquela região é mais difícil de representar com as features atuais.

## 19. Temperatura média real vs prevista por tipo de área

Este gráfico mostra se o modelo consegue reproduzir as diferenças médias entre os tipos de área.

Ele é útil para observar se o modelo mantém coerência climática entre regiões urbanas, litorâneas, serranas e interiores.

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

media_tipo_area_amanha = spark.sql(f"""
    SELECT
        tipo_area_idx,
        ROUND(AVG({coluna_alvo_amanha}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media
    FROM pred_amanha_ctx
    GROUP BY tipo_area_idx
    ORDER BY tipo_area_idx
""")

media_tipo_area_amanha.show(truncate=False)

In [ ]:
media_tipo_area_amanha.createOrReplaceTempView("media_tipo_area_amanha")

media_tipo_area_amanha_rows = spark.sql("""
    SELECT
        tipo_area_idx,
        temperatura_real_media,
        temperatura_prevista_media
    FROM media_tipo_area_amanha
    ORDER BY tipo_area_idx
""").collect()

media_tipo_area_amanha_plot = {
    "tipo_area_idx": [row["tipo_area_idx"] for row in media_tipo_area_amanha_rows],
    "temperatura_real_media": [row["temperatura_real_media"] for row in media_tipo_area_amanha_rows],
    "temperatura_prevista_media": [row["temperatura_prevista_media"] for row in media_tipo_area_amanha_rows]
}

In [ ]:
media_tipo_area_amanha.createOrReplaceTempView("media_tipo_area_amanha")

media_tipo_area_long = spark.sql("""
    SELECT
        tipo_area_idx,
        'temperatura_real_media' AS tipo,
        temperatura_real_media AS temperatura_media
    FROM media_tipo_area_amanha

    UNION ALL

    SELECT
        tipo_area_idx,
        'temperatura_prevista_media' AS tipo,
        temperatura_prevista_media AS temperatura_media
    FROM media_tipo_area_amanha
""")

media_tipo_area_long.createOrReplaceTempView("media_tipo_area_long")

media_tipo_area_long_rows = spark.sql("""
    SELECT
        tipo_area_idx,
        tipo,
        temperatura_media
    FROM media_tipo_area_long
    ORDER BY tipo_area_idx, tipo
""").collect()

media_tipo_area_long_plot = {
    "tipo_area_idx": [row["tipo_area_idx"] for row in media_tipo_area_long_rows],
    "tipo": [row["tipo"] for row in media_tipo_area_long_rows],
    "temperatura_media": [row["temperatura_media"] for row in media_tipo_area_long_rows]
}

fig = px.bar(
    media_tipo_area_long_plot,
    x="tipo_area_idx",
    y="temperatura_media",
    color="tipo",
    barmode="group",
    text="temperatura_media",
    title="Temperatura média real vs prevista por tipo de área — amanhã",
    labels={
        "tipo_area_idx": "Tipo de área",
        "temperatura_media": "Temperatura média (°C)",
        "tipo": "Série"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Espera-se que regiões de `serra_altitude` apresentem temperaturas médias menores, por causa da influência da altitude.

Áreas `urbano_metropolitano` podem apresentar médias mais altas, possivelmente relacionadas à concentração urbana e ao efeito de ilha de calor.

Regiões de `litoral` tendem a ter comportamento mais estável em alguns contextos, devido à influência oceânica e à umidade.

## 20. Erro por macro região

Além do tipo de área, também é útil observar o desempenho por macro região aproximada do estado de São Paulo.

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

erro_macro_amanha = spark.sql(f"""
    SELECT
        macro_regiao_sp_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG({coluna_alvo_amanha}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_amanha_ctx
    GROUP BY macro_regiao_sp_idx
    ORDER BY macro_regiao_sp_idx
""")

erro_macro_amanha.show(truncate=False)

In [ ]:
erro_macro_amanha.createOrReplaceTempView("erro_macro_amanha")

erro_macro_amanha_rows = spark.sql("""
    SELECT
        macro_regiao_sp_idx,
        mae
    FROM erro_macro_amanha
    ORDER BY macro_regiao_sp_idx
""").collect()

erro_macro_amanha_plot = {
    "macro_regiao_sp_idx": [row["macro_regiao_sp_idx"] for row in erro_macro_amanha_rows],
    "mae": [row["mae"] for row in erro_macro_amanha_rows]
}

fig = px.bar(
    erro_macro_amanha_plot,
    x="macro_regiao_sp_idx",
    y="mae",
    text="mae",
    title="Erro médio absoluto por macro região — previsão de amanhã",
    labels={
        "macro_regiao_sp_idx": "Macro região",
        "mae": "MAE (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 21. Relação entre altitude e erro

A altitude é uma variável importante para temperatura.

Este gráfico ajuda a observar se o modelo tem maior dificuldade em regiões mais altas, como áreas de serra.

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

altitude_erro_rows = spark.sql("""
    SELECT
        altitude_imputado,
        tipo_area_idx,
        erro_absoluto
    FROM pred_amanha_ctx
    WHERE altitude_imputado IS NOT NULL
      AND tipo_area_idx IS NOT NULL
      AND erro_absoluto IS NOT NULL
""").collect()

altitude_erro_plot = {
    "altitude_imputado": [row["altitude_imputado"] for row in altitude_erro_rows],
    "tipo_area_idx": [row["tipo_area_idx"] for row in altitude_erro_rows],
    "erro_absoluto": [row["erro_absoluto"] for row in altitude_erro_rows]
}

In [ ]:
fig = px.scatter(
    altitude_erro_plot,
    x="altitude_imputado",
    y="erro_absoluto",
    color="tipo_area_idx",
    opacity=0.5,
    title="Relação entre altitude e erro absoluto — previsão de amanhã",
    labels={
        "altitude_imputado": "Altitude (m)",
        "erro_absoluto": "Erro absoluto (°C)",
        "tipo_area_idx": "Tipo de área"
    }
)

fig.show()

### Insight esperado

Caso os erros aumentem em altitudes maiores, isso pode indicar que regiões serranas possuem comportamento térmico mais específico.

Mesmo com a variável `altitude` presente no modelo, fatores locais como relevo, cobertura vegetal e massas de ar podem influenciar a temperatura de maneira mais complexa.

## 22. Salvamento das predições, métricas e importâncias

As saídas do notebook são salvas para comparação posterior com outros modelos, como Regressão Linear e Redes Neurais.

In [ ]:
pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")
pred_semana_ctx.createOrReplaceTempView("pred_semana_ctx")

spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        {coluna_alvo_amanha},
        prediction,
        erro_absoluto
    FROM pred_amanha_ctx
""").write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_rf_amanha"
)

spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        {coluna_alvo_semana},
        prediction,
        erro_absoluto
    FROM pred_semana_ctx
""").write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_rf_semana"
)

metricas_rf.write.mode("overwrite").parquet(
    f"{resultados_path}/metricas_random_forest"
)

importancias_amanha.write.mode("overwrite").parquet(
    f"{resultados_path}/importancias_rf_amanha"
)

importancias_semana.write.mode("overwrite").parquet(
    f"{resultados_path}/importancias_rf_semana"
)

print("Predições, métricas e importâncias salvas com sucesso.")

## 23. Salvamento dos modelos treinados

Os modelos são salvos para que possam ser reutilizados sem necessidade de novo treinamento.

In [ ]:
modelo_rf_amanha.write().overwrite().save(
    f"{modelos_path}/random_forest_amanha"
)

modelo_rf_semana.write().overwrite().save(
    f"{modelos_path}/random_forest_semana"
)

print("Modelos Random Forest salvos com sucesso.")

## 24. Conclusão

Neste notebook foram treinados dois modelos **Random Forest Regressor** com **PySpark MLlib**:

- um modelo para prever a temperatura média de amanhã;
- um modelo para prever a temperatura média dos próximos 7 dias.

As bases utilizadas vieram diretamente do pré-processamento, já com:

- split temporal;
- imputação de valores ausentes;
- indexação de variáveis categóricas;
- features temporais de defasagem e janelas móveis.

A avaliação foi feita com **MAE**, **RMSE** e **R²**.

Além disso, foram gerados gráficos com Plotly para analisar:

- comparação geral das métricas;
- importância das variáveis;
- erro por tipo de área;
- real vs previsto por tipo de área;
- erro por mês;
- temperatura média real vs prevista;
- erro por macro região;
- relação entre altitude e erro.

A análise por `tipo_area` permite observar diferenças entre áreas urbanas/metropolitanas, litorâneas, serranas e interiores.

O Random Forest não entende sequência temporal automaticamente. Por isso, as features de defasagem e janelas móveis criadas no pré-processamento foram fundamentais para fornecer contexto temporal ao modelo.